<a href="https://colab.research.google.com/github/behan02/langchain-tutorial/blob/main/chapters/05-structured_output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Structured Output using Pydantic and TypedDict

In [ ]:
!pip install -qU \
  langchain-core \
  langchain-google-genai \
  langchain-community

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

os.environ["GOOGLE_API_KEY"] = ""

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0)

## Guiding using Prompts

In [15]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant"),
    ("user","Tell me a joke. Generate the output in key-value pair format with the following keys: setup, punchline")
])

result = llm.invoke(prompt_template.format_messages())
result.content

'```json\n{\n  "setup": "Why don\'t scientists trust atoms?",\n  "punchline": "Because they make up everything!"\n}\n```'

## Using Pydantic Models

**Pydantic** is a data validation library. It uses Python classes (`BaseModel`) to enforce types at **runtime**. If the data doesn't match the schema, it throws an error. Use this when you need strict validation and want to treat data as objects.

In [13]:
from pydantic import BaseModel, Field

class my_schema(BaseModel):
  setup: str = Field(description="The setup for the joke")
  punchline: str = Field(description="The punchline for the joke")

llm_structured_output = llm.with_structured_output(my_schema)

result = llm_structured_output.invoke("Tell me a joke")
result

my_schema(setup="Why don't scientists trust atoms?", punchline='Because they make up everything!')

## Using TypedDict

**TypedDict** is a type hint for dictionaries. It provides **static type checking** (like in your IDE) but does not enforce types at runtime. Use this when you want simple dictionary outputs without the overhead of validation classes.

In [14]:
from typing import TypedDict

class my_schema_td(TypedDict):
  setup: str
  punchline: str

llm_structured_output_td = llm.with_structured_output(my_schema_td)

result = llm_structured_output_td.invoke("Tell me a joke")
result

{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything!'}